# Nombre de paires de gènes avec des valeurs TE nulles

Ce notebook compte, dans `Données/lamme_training.csv`, les paires de gènes pour lesquelles les deux gènes ont simultanément une valeur égale à 0 pour `TE_Density`, `TE_Coverage` et `TE_Number`.

In [1]:
from pathlib import Path
import pandas as pd

## Chargement des données

In [2]:
dataset_path = Path('Données/lamme_training.csv')

if not dataset_path.exists():
    raise FileNotFoundError(f'Fichier introuvable : {dataset_path.resolve()}')

df = pd.read_csv(dataset_path)
print(f'{len(df):,} paires de gènes chargées.')

444 paires de gènes chargées.


In [3]:
descriptors = {
    'TE_Density': ('TE_Density_G1', 'TE_Density_G2'),
    'TE_Coverage': ('TE_Coverage_G1', 'TE_Coverage_G2'),
    'TE_Number': ('TE_Number_G1', 'TE_Number_G2'),
}

required_columns = [column for pair in descriptors.values() for column in pair]
missing_columns = sorted(set(required_columns) - set(df.columns))
if missing_columns:
    raise KeyError(f'Colonnes manquantes : {missing_columns}')

df[required_columns].head()

,TE_Density_G1,TE_Density_G2,TE_Coverage_G1,TE_Coverage_G2,TE_Number_G1,TE_Number_G2
0,0.167056,0.193424,1.318826,5.777292,1.0,1.0
1,0.167056,0.745712,1.318826,29.346681,1.0,3.0
2,0.000000,0.000000,0.000000,0.000000,0.0,0.0
3,0.200080,0.000000,15.273775,0.000000,1.0,0.0
4,0.200080,0.000000,15.273775,0.000000,1.0,0.0


## Comptage des paires dont les deux valeurs sont nulles

Les lignes contenant une valeur manquante pour l'une des deux composantes ne sont pas comptées comme des paires nulles.

In [4]:
results = []

for descriptor, (gene1_column, gene2_column) in descriptors.items():
    valid_values = df[[gene1_column, gene2_column]].notna().all(axis=1)
    both_zero = (df[gene1_column] == 0) & (df[gene2_column] == 0)
    zero_pair_mask = valid_values & both_zero

    results.append({
        'Descripteur': descriptor,
        'Nombre total de paires': len(df),
        'Paires analysables': int(valid_values.sum()),
        'Paires avec G1 = G2 = 0': int(zero_pair_mask.sum()),
        'Pourcentage des paires analysables': round(100 * zero_pair_mask.sum() / valid_values.sum(), 2)
        if valid_values.sum() else 0.0,
        'Valeurs manquantes': int((~valid_values).sum()),
    })

results_df = pd.DataFrame(results)
results_df

,Descripteur,Nombre total de paires,Paires analysables,Paires avec G1 = G2 = 0,Pourcentage des paires analysables,Valeurs manquantes
0,TE_Density,444,444,139,31.31,0
1,TE_Coverage,444,444,139,31.31,0
2,TE_Number,444,444,139,31.31,0


## Résultat direct

In [5]:
for _, row in results_df.iterrows():
    print(f"{row['Descripteur']}: {row['Paires avec G1 = G2 = 0']} paires")

TE_Density: 139 paires
TE_Coverage: 139 paires
TE_Number: 139 paires
